# オープンキャンパス管理システム — Colab + ngrok 起動ノートブック

Google Colab 上でアプリを起動し、**ngrok** で外部公開URLを発行します。上から順に実行してください。

> **事前準備**: ngrok は無料でも authtoken が必要です。
> https://dashboard.ngrok.com/get-started/your-authtoken で取得しておいてください（無料登録）。

> **データ永続化**: 既定では DB は一時領域に置かれ、ランタイム切断で消えます。
> 当日の履歴・待機列を残したい場合は **セル2でDriveマウントを有効**にしてください。

## 1. 必要ライブラリのインストール

In [ ]:
!pip install flask flask-sqlalchemy pytz pyngrok -q
print("✅ ライブラリのインストール完了")

## 2. データ永続化の設定（任意）

`USE_DRIVE = True` にすると Google Drive をマウントし、DB を Drive 上に置いて
ランタイム切断後もデータを保持します。`False` の場合は一時領域（切断で消える）。

In [ ]:
import os

USE_DRIVE = False  # ← データを永続化したい場合は True にする

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_DIR = '/content/drive/MyDrive/opencampus_system'
    os.makedirs(DB_DIR, exist_ok=True)
    os.environ['DB_PATH'] = os.path.join(DB_DIR, 'opencampus.db')
    print('✅ DBをDriveに保存します:', os.environ['DB_PATH'])
else:
    print('ℹ️ DBは一時領域に保存されます（ランタイム切断で消えます）')

## 3. リポジトリの取得（最新コードを GitHub から取得）

初回は `git clone`、2回目以降は `git pull` で最新に更新します。

In [ ]:
REPO_URL = "https://github.com/noirelumiere00/TIUkanri.git"
BRANCH = "claude/admiring-tesla-Hy3l8"
PROJECT_DIR = "/content/TIUkanri"

if not os.path.exists(PROJECT_DIR):
    !git clone -b $BRANCH $REPO_URL $PROJECT_DIR
else:
    !cd $PROJECT_DIR && git pull origin $BRANCH

os.chdir(PROJECT_DIR)
print("✅ 作業ディレクトリ:", os.getcwd())
print("ファイル一覧:", os.listdir('.'))

## 4. ngrok の authtoken を設定

実行するとトークンの入力欄が出ます（入力内容は画面に表示されません）。

In [ ]:
import getpass
from pyngrok import ngrok

token = getpass.getpass('ngrok authtoken を貼り付けて Enter: ')
ngrok.set_auth_token(token)
print("✅ authtoken を設定しました")

## 5. アプリ起動 + 公開URLの発行

公開URLが表示されたら、それをブラウザで開いてください。
**このセルは起動中ずっと実行されたままになります**（停止するには停止ボタン）。

> 複数スタッフで使う場合、URLが漏れた人は誰でも操作できます。
> 簡易的に保護したい場合は、起動前のセルで
> `os.environ['APP_PASSWORD'] = '合言葉'` を設定してください（BASIC認証が有効になります）。

In [ ]:
import sys
from pyngrok import ngrok

# 既存のトンネル / キャッシュをクリア（再実行時の二重起動防止）
ngrok.kill()
for m in ['app', 'models', 'config']:
    sys.modules.pop(m, None)

PORT = 5000
public_url = ngrok.connect(PORT)
print("=" * 50)
print("🌐 公開URL:", public_url.public_url)
print("=" * 50)

from app import app
# Colab ではリローダーを無効にして起動する
app.run(port=PORT, use_reloader=False)

## 6. DBバックアップ（任意・別タブで随時実行）

起動中でも、このセルだけ別途実行すると現在のDBを Drive にコピーできます。
（セル2で `USE_DRIVE=True` にしていれば、そもそも Drive 上に保存されています）

In [ ]:
import shutil, os, datetime

src = os.environ.get('DB_PATH', os.path.join(PROJECT_DIR, 'opencampus.db'))
backup_dir = '/content/drive/MyDrive/opencampus_system/backups'
try:
    os.makedirs(backup_dir, exist_ok=True)
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    dst = os.path.join(backup_dir, f'opencampus_{stamp}.db')
    shutil.copy(src, dst)
    print('✅ バックアップ完了:', dst)
except Exception as e:
    print('⚠️ バックアップ失敗（先にセル2でDriveをマウントしてください）:', e)

---
### 補足

- 公開URLは無料プランでは起動のたびに変わります。
- トンネルを手動で閉じるには `from pyngrok import ngrok; ngrok.kill()` を実行します。
- 制限時間や長時間待機の閾値は環境変数で変更できます（起動前に設定）:
  `os.environ['TIME_LIMIT_MIN'] = '20'` / `os.environ['LONG_WAIT_MIN'] = '40'`
- ブース名は管理画面の「⚙ ブース管理」から追加・改名・削除できます。